# BLOK · Partie 3 · Smart RFQ

Objectif : transformer un message WhatsApp (« bjr il me faut 200 sacs de ciment 42,5 ») en lignes de devis : `variant_id`, quantité, indice de confiance, ou un cas explicite « absent du catalogue » / « ambigu, à faire préciser ».

Approche retenue : des règles appuyées sur le catalogue d'abord, un LLM seulement en recours.
- Le catalogue ne compte que 26 produits, avec des caractéristiques très structurées (42.5, HA 12, 1,5 L, A4). Des règles suffisent pour la majorité des messages. Elles sont gratuites, instantanées (moins d'une milliseconde par message), reproductibles, et chaque décision s'explique.
- Le LLM est utile pour ce que les règles gèrent mal : synonymes imprévus, phrases libres, fautes lourdes. On ne l'appelle que pour les messages à faible confiance (section 5).

Fonctionnement du moteur (`src/smart_rfq.py`, environ 150 lignes) :
1. Normaliser : minuscules, sans accents, `42,5` devient `42.5`, `x4` devient `x 4`.
2. Trouver la famille de produits : mots du message comparés aux mots du catalogue (exact, par préfixe comme `sécu` pour `sécurité`, ou flou pour les fautes de frappe). Les mots rares pèsent plus (pondération IDF).
3. Départager par les caractéristiques : `42.5`, `HA12`, `8mm`, `1.5L`, `A4`, `85A`. Une caractéristique incompatible (`50cl` alors que le catalogue n'a que du 1,5 L, `CO2 2kg`, fer de 14) donne absent du catalogue. Plusieurs produits encore possibles donnent ambigu.
4. Lire la quantité : nombre après `x`, sinon devant une unité (sacs, barres, paires…), sinon le nombre restant ; « une centaine » vaut 100, avec une confiance réduite.
5. Indice de confiance : part de 0,95, baisse si la correspondance est floue, si un seul mot est reconnu, si la quantité est approximative ou absente.

In [16]:
import sys, time
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from smart_rfq import Catalogue, extraire

catalogue = Catalogue(pd.read_csv(ROOT / "data" / "product_variants.csv"))
rfq = pd.read_csv(ROOT / "data" / "rfq_requests.csv")
pd.set_option("display.max_colwidth", 70)

## 1. Résultats du prototype sur les 30 messages

In [17]:
def formater(resultats):
    """Format comparable à l'annotation : '101:200', '122:40 + 125:40', 'absent' ou 'ambigu'."""
    morceaux = []
    for r in resultats:
        if r["statut"].startswith("absent"):
            morceaux.append("absent")
        elif r["statut"].startswith("ambigu"):
            morceaux.append("ambigu")
        else:
            q = "?" if r["quantite"] is None else f"{r['quantite']:g}"
            morceaux.append(f"{r['variant_id']}:{q}")
    return " + ".join(morceaux)

debut = time.perf_counter()
sorties = [extraire(m, catalogue) for m in rfq.client_message]
duree_ms = (time.perf_counter() - debut) * 1000 / len(rfq)

rfq["prediction"] = [formater(s) for s in sorties]
rfq["statut"] = [" + ".join(r["statut"] for r in s) for s in sorties]
rfq["confiance"] = [min(r["confiance"] for r in s) for s in sorties]
rfq["explication"] = [" | ".join(r["motif"] for r in s) for s in sorties]
print(f"Temps moyen : {duree_ms:.2f} ms par message")
rfq[["request_id", "client_message", "prediction", "statut", "confiance", "explication"]]

Temps moyen : 1.99 ms par message


,request_id,client_message,prediction,statut,confiance,explication
0,RFQ-001,"Bjr svp il me faut 200 sacs de ciment 42,5 pour lundi",101:200,ok,0.95,Ciment CPJ 42.5 - sac 50 kg
1,RFQ-002,ciment cpj 32 - 80 sacs,102:80,ok,0.95,Ciment CPJ 32.5 - sac 50 kg
2,RFQ-003,Fer de 10 x 150 barres,104:150,ok,0.95,Fer à béton HA 10 - barre 12 m
3,RFQ-004,fer HA12 300,105:300,ok,0.95,Fer à béton HA 12 - barre 12 m
4,RFQ-005,"Barres de fer 8mm, une centaine",103:100,ok,0.75,Fer à béton HA 8 - barre 12 m
5,RFQ-006,tôles alu 3m 6/10 : 60,106:60,ok,0.95,Tôle bac alu 6/10 - 3 m
6,RFQ-007,Peinture blanche à eau 10 seaux de 25kg,107:10,ok,0.95,Peinture à eau blanche - seau 25 kg
7,RFQ-008,2 camions de sable lagunaire,108:2,ok,0.95,Sable de lagune - camion 10 m3
8,RFQ-009,50 rames de papier A4,109:50,ok,0.80,Ramette papier A4 80 g - 500 feuilles
9,RFQ-010,Toner 85A x4 pour imprimante HP,110:4,ok,0.95,Toner HP 85A


## 2. Annotation manuelle

Les bonnes réponses ont été annotées à la main dans `annotations/annotations_manuelles.csv`, sans regarder les prédictions. Format : `101:200` (variant_id:quantité), `122:40 + 125:40` pour plusieurs produits, `absent` ou `ambigu`.

In [18]:
annot = pd.read_csv(ROOT / "annotations" / "annotations_manuelles.csv", dtype=str).fillna("")
if (annot.attendu.str.strip() == "").any():
    print(f"Annotation incomplète : {(annot.attendu.str.strip() == '').sum()} messages sans réponse attendue.")
else:
    def normal(s):
        return " + ".join(sorted(x.strip().lower() for x in s.split("+")))
    eval_ = rfq.merge(annot[["request_id", "attendu", "commentaire"]], on="request_id")
    eval_["correct"] = eval_.prediction.map(normal) == eval_.attendu.map(normal)
    eval_["bon_produit"] = (eval_.prediction.str.replace(r":[\d.?]+", "", regex=True).map(normal)
                            == eval_.attendu.str.replace(r":[\d.?]+", "", regex=True).map(normal))
    print(f"Taux de bonnes réponses (produit ET quantité ET statut) : {eval_.correct.mean():.0%} ({eval_.correct.sum()}/{len(eval_)})")
    print(f"Bon produit ou bon statut, quantité ignorée : {eval_.bon_produit.mean():.0%}")
    print(f"Confiance moyenne : {eval_[eval_.correct].confiance.mean():.2f} si correct, "
          f"{eval_[~eval_.correct].confiance.mean() if (~eval_.correct).any() else float('nan'):.2f} si erreur")
    display(eval_[~eval_.correct][["request_id", "client_message", "attendu", "prediction", "confiance", "explication"]])

Taux de bonnes réponses (produit ET quantité ET statut) : 90% (27/30)
Bon produit ou bon statut, quantité ignorée : 90%
Confiance moyenne : 0.88 si correct, 0.72 si erreur


,request_id,client_message,attendu,prediction,confiance,explication
4,RFQ-005,"Barres de fer 8mm, une centaine",ambigu,103:100,0.75,Fer à béton HA 8 - barre 12 m
28,RFQ-029,"Câble électrique 2,5mm² 3 rouleaux",absent,120:3,0.80,Papier hygiénique - colis de 48 rouleaux
29,RFQ-030,Fer à béton 14 x 50,ambigu,absent,0.60,"caractéristique 14 inconnue pour : Fer à béton HA 8 - barre 12 m, ..."


## 3. Trois erreurs expliquées

Erreur 1 : RFQ-OO5, j'ai annoté ambigu car `une centaine` ne designe pas une quantité fixe. Le modèle à répondu 103:100 car le produits est bien identifié et la quantité fait penser au nombre 100. Par conséquent le niveau de confiance est en dessous des 0.75.

Erreur 2 : RFQ-029, le produit est absent du catalogue mais le modele le réconnait quand même à cause du termes `rouleaux`, raison pour laquelle il identifie le produit `papier hygiénique` Donc il faut passer en révue la ligne d'un produit surtout si le mot ayant permis d'identifier la ligne ne se trouve pas dans le nom du produits

Erreur 3 : RFQ-030, ICI il n'y a pas véritablement de mauvaise réponse, juste qu'il serait bien de demander au client plus de précision

Limite de la mesure : les règles ont été conçues en ayant les 30 messages sous les yeux. Le score est donc optimiste : la vraie mesure demanderait de nouveaux messages, jamais vus.

## 4. Recours au LLM et coût

Pour les messages à faible confiance, on peut appeler un LLM avec le catalogue dans le prompt et une sortie JSON imposée. Le prompt interdit d'inventer : tout produit hors liste doit revenir en `absent`. Le code n'est pas exécuté par défaut (il faut une clé API). Les prix utilisés pour l'estimation sont des hypothèses à vérifier sur la grille tarifaire.

In [19]:
UTILISER_LLM = False

liste_catalogue = "\n".join(f"{p['id']} | {p['nom']}" for p in catalogue.produits)
PROMPT = f"""Tu convertis une demande client en lignes de devis.
Catalogue (variant_id | produit) :
{liste_catalogue}

Règles : n'utilise que les variant_id ci-dessus. Si le produit n'est pas au catalogue, statut "absent".
Si plusieurs produits sont possibles (ex. ciment sans précision 32.5 / 42.5), statut "ambigu".
Réponds uniquement en JSON : [{{"variant_id": int|null, "quantite": number|null, "statut": "ok"|"absent"|"ambigu", "confiance": 0-1}}]
Message : """

def extraire_llm(message, modele="claude-haiku-5-5"):
    import json, anthropic
    reponse = anthropic.Anthropic().messages.create(
        model=modele, max_tokens=300, messages=[{"role": "user", "content": PROMPT + message}])
    return json.loads(reponse.content[0].text)

if UTILISER_LLM:
    douteux = rfq[rfq.confiance < 0.7]
    douteux["llm"] = douteux.client_message.map(extraire_llm)
    display(douteux[["client_message", "prediction", "llm"]])

In [20]:
tokens_entree = len(PROMPT) / 4 + 15
tokens_sortie = 80
PRIX_ENTREE_USD_MTOK, PRIX_SORTIE_USD_MTOK = 1.0, 5.0
USD_EN_FCFA = 600
part_envoyee_au_llm = (rfq.confiance < 0.7).mean()

for nom, part in [("Tout au LLM", 1.0), ("Hybride (LLM si confiance < 0.7)", part_envoyee_au_llm)]:
    cout_usd = 1000 * part * (tokens_entree * PRIX_ENTREE_USD_MTOK + tokens_sortie * PRIX_SORTIE_USD_MTOK) / 1e6
    print(f"{nom:35s} {1000 * part:5.0f} appels/mois  ≈ {cout_usd:.2f} USD ≈ {cout_usd * USD_EN_FCFA:.0f} FCFA / mois")
print(f"Prompt ≈ {tokens_entree:.0f} tokens en entrée, {tokens_sortie} en sortie ; latence LLM de l'ordre d'une à quelques secondes contre {duree_ms:.2f} ms pour les règles.")

Tout au LLM                          1000 appels/mois  ≈ 0.76 USD ≈ 454 FCFA / mois
Hybride (LLM si confiance < 0.7)      133 appels/mois  ≈ 0.10 USD ≈ 61 FCFA / mois
Prompt ≈ 357 tokens en entrée, 80 en sortie ; latence LLM de l'ordre d'une à quelques secondes contre 1.99 ms pour les règles.


Lecture : même en envoyant tout au LLM, le coût reste de l'ordre d'un dollar par mois pour 1 000 messages, et l'hybride le divise encore par sept : le coût n'est pas le frein. Les vrais sujets sont la latence, la reproductibilité et le contrôle des réponses inventées. D'où l'architecture hybride : règles d'abord, LLM pour les cas douteux, et validation humaine pour tout ce qui n'est pas « ok ».

## 5. Réflexion (3.B) : comment un assistant IA peut donner un chiffre faux mais crédible

Un assistant peut donner un chiffre faux mais crédible s’il lit un champ peu fiable, s’il oublie une règle métier ou s’il ne précise pas le périmètre de sa réponse. Il doit donc interroger des vues validées, afficher la requête et le périmètre de chaque chiffre, et des contrôles automatiques doivent tourner avant toute publication